In [ ]:
import sys
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import partipy as pt
import matplotlib.pyplot as plt

In [ ]:
sys.path.append("..")
from scripts.utils import *

## Read in your adata 

In [ ]:
counts = pd.read_csv("../data/processed/output_counts.csv", index_col=0)
meta   = pd.read_csv("../data/processed/output_metadata.csv", index_col=0)
adata  = sc.AnnData(X=counts.T, obs=meta)

In [ ]:
adata

## Preprocessing Assumptions

This pipeline assumes that the dataset has already undergone initial quality control (QC) and preprocessing steps, including:

- Removal of low-quality cells
- Filtering of doublets/multiplets
- Correction or exclusion of cells with high mitochondrial contamination
- Removal of excessive ribosomal contamination
- Any other standard sample-specific QC filtering

> **Note:** If these preprocessing steps have **not** been completed yet, please perform them first before proceeding with the downstream analysis in this pipeline.

In [ ]:
# QC codes here if needed

## Raw Counts Check

This pipeline assumes that `adata.X` contains **raw counts** (unnormalized, untransformed integer counts).  
If your `adata.X` has already been normalized or log-transformed, the results downstream may be incorrect.

Run the cell below to verify:

In [ ]:
check_raw_integers_in_adataX(adata)

## Gene exclusion list for PCA

Genes to optionally exclude before PCA — they can drive PCs for technical
or biological reasons unrelated to cell identity:

- **Mitochondrial genes** — high MT expression flags low-quality cells;
  residual stressed cells can dominate early PCs (`MT-` / `mt-` prefix)
- **Ribosomal protein genes** — vary with sequencing depth and cell size
  rather than cell type (`RPS*`, `RPL*`)
- **Sex-chromosome genes** — donor sex drives a PC in mixed-sex cohorts
  (e.g. `XIST`, `DDX3Y`, `UTY`, `KDM5D`)
- **Study-specific genes** — batch outliers, dissociation-stress genes
  (`JUN`, `FOS`), haemoglobin genes (`HBA1`, `HBB`) from RBC contamination,
  or any gene flagged during QC

In [ ]:
QC_genes = pd.read_csv("../data/accessories/QC_genes.txt",sep="\t")
QC_genes = QC_genes.iloc[:, 0].tolist()

## Single-Cell Preprocessing Pipeline

This step performs **standard single-cell RNA-seq preprocessing** on the AnnData object, including:

1. **Normalization** – Total counts per cell (library size correction)
2. **Log-transformation** – `log1p` to stabilize variance
3. **Highly Variable Gene (HVG) selection** – Focus on biologically meaningful genes
4. **Gene exclusion** – Optionally remove:
   - Quality-associated genes (e.g., mitochondrial, ribosomal)
   - Custom genes (e.g., sex-linked, batch-effect-prone, or study-specific confounders)
5. **Scaling** – Z-score normalization (stored in `adata.layers["z_scaled"]`)
6. **Dimensionality reduction** – PCA (default: 50 PCs)
7. **Batch correction (optional)** – Harmony integration (if `apply_harmony=True`)

In [ ]:
help(preprocess_adata)

In [ ]:
preprocess_adata(adata, exclude_quality_genes=True, custom_exclude_genes=QC_genes, n_pcs=50, pca_seed=123)

In [ ]:
adata

## Visualize Cells Across Multiple Principal Components

In [ ]:
sc.pl.pca_scatter(adata, components=['1,2', '1,3', '2,3'])

## Decide the Number of PCs for Downstream Analysis

Next, decide how many **principal components (PCs)** to carry forward for downstream steps such as:

Choosing too **few PCs** may miss meaningful biological variation, while choosing too **many** may introduce noise or technical effects.

A common way to decide this is to look at:

- the **elbow plot** / variance explained per PC
- whether later PCs capture **real biology** or mostly **technical noise**
- whether specific PCs are dominated by unwanted signals such as:
  - mitochondrial genes
  - ribosomal genes
  - cell cycle effects
  - batch / sample effects

> **Rule of thumb:** Keep enough PCs to capture meaningful structure, but avoid carrying obvious noise into downstream analysis.

In [ ]:
pt.compute_shuffled_pca(adata, mask_var="highly_variable")
pt.plot_shuffled_pca(adata)

In [ ]:
n_dims = 8

## Compute Archetype Selection Metrics (ParTI-py)

Now we use **ParTI-py** to compute archetype selection metrics across a range of archetype numbers (`k`).

### What This Step Does

1. **Sets the PCA embedding** for ParTI-py using the first `n_dims` PCs
2. **Computes selection metrics** for a range of `k` values (number of archetypes)
   - Metrics help decide the **optimal number of archetypes** to use downstream


- Make sure you've already decided on the **number of PCs (`n_dims`)** to use (see previous cell).
- The selection metrics (e.g., explained variance, residual sum of squares) will be stored in `adata` for downstream visualization.
- After running this, plot the metrics to identify the optimal `k` where adding more archetypes yields diminishing returns.

In [ ]:
adata = get_partipy_selection_metrics(adata, n_dims, k_min=2, k_max=7)

## Archetype Number Selection (Diagnostics)

Before choosing a final number of archetypes, generate diagnostic plots to evaluate **stability**, **variance explained**, and **statistical significance** across multiple candidate values of optimal number of archetypes

In [ ]:
plots_for_n_archetypes_selection(adata, n_archetype_range=range(3, 8))

In [ ]:
plot_archetypes_3D_range(adata, n_archetype_range=range(4, 8))

## Choose the Number of Archetypes

Based on the diagnostic plots above, choose the number of archetypes (`n_archetypes`) to use for downstream analysis.

In [ ]:
n_archetypes = 3

## Assign Top Cells to Each Archetype

Identify the `top_n` cells closest to each archetype in PCA space and store the assignment in `adata.obs["archetype"]`.

In [ ]:
adata = get_top_cells_per_archetype(adata, n_archetypes= n_archetypes, n_dims= n_dims, top_n=100)

## Visualize Top Archetype Cells

Plot the assigned top cells for each archetype on the selected PCA dimensions.

In [ ]:
plot_top_cells_per_archetype(adata, dims=(0,2))

## Differential Expression per Archetype

Once cells have been assigned to their dominant archetype, run a **1-vs-rest differential expression analysis** to identify the marker genes that define each archetype.

**Prerequisites:** adata.obs must contain an "archetype" column (run get_top_cells_per_archetype first). Cells assigned 0 are treated as unassigned and excluded.

In [ ]:
deg_dict = run_deg_per_archetype(adata)

In [ ]:
deg_dict

## Pairwise Differential Expression Between Archetypes

While 1-vs-rest DEG analysis identifies broad markers for each archetype, **pairwise comparisons** reveal genes that specifically distinguish one archetype from another — useful for understanding fine-grained biological differences between closely related archetypes.

In [ ]:
pairwise_deg_dict = run_pairwise_deg_per_archetype(adata, lfc_threshold=1.0, pval_threshold=0.05)

In [ ]:
pairwise_deg_dict

## Strict Archetype Marker Genes

After running both 1-vs-rest and pairwise DEG analyses, intersect the results to obtain a **strict set of marker genes** for each archetype — genes that are not just generally distinctive, but specifically upregulated against **every** other archetype.

In [ ]:
strict_genes_df   = get_strict_archetype_genes(deg_dict, pairwise_deg_dict)

In [ ]:
strict_genes_df

## GO Enrichment Analysis

At this point, we have identified **marker genes for each archetype** — genes that are 
differentially expressed in one archetype relative to all others. But a list of gene names 
doesn't tell us much on its own. To interpret *what each archetype is doing biologically*, 
we map these marker genes onto known gene ontology (GO) terms.

We use **Enrichr** (via `gseapy`) to test whether our marker genes are statistically 
overrepresented in any biological processes, molecular functions, or cellular components 
relative to what we'd expect by chance. Critically, we set the **background to all genes 
detected in our dataset** — not the entire genome — because we can only have detected 
enrichment for genes we actually measured.

A significant GO term tells us that an archetype is not just defined by random genes, 
but by genes that share a coherent biological role. This is how we go from 
*"archetype 2 exists"* to *"archetype 2 represents cells engaged in synaptic transmission"*.

In [ ]:
go_results = run_go_analysis(deg_dict, adata, organism="mouse", n_top_genes=200)

In [ ]:
go_results[1].shape

In [ ]:
go_results[2].shape

In [ ]:
go_results[3].shape

In [ ]:
go_results

## GO Analysis with "strict" archetype gene list

In [ ]:
strict_go_results = run_strict_go_analysis(strict_genes_df, adata, organism="mouse")

In [ ]:
strict_go_results